# Creating the CNN
We will be using a Convolutional Neural Network to predict if audio has been deepfaked.

In [1]:
!pip install tensorflow

In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt

# Load your processed data
data = np.load('processed_data.npz')
X_train, y_train = data['x_train'], data['y_train']
X_val, y_val = data['x_val'], data['y_val']
X_test, y_test = data['x_test'], data['y_test']

print(f"Ready to train on {X_train.shape[0]} samples of shape {X_train.shape[1:]}")

Ready to train on 13956 samples of shape (128, 44, 1)


# Create the Model



In [3]:
# model = models.Sequential([
#     # Layer 1: Input Layer + Convolution
#     # 32 filters of size 3x3 to find basic edge/texture patterns
#     layers.Conv2D(32, (3, 3), activation='relu', input_shape=(128, 44, 1)),
#     layers.MaxPooling2D((2, 4)), # Shrinks the image to focus on key features
    
#     # Layer 2: Deeper features
#     layers.Conv2D(64, (3, 3), activation='relu'),
#     layers.MaxPooling2D((2, 2)),
    
#     # Layer 3: Flattening and Dense Layers
#     layers.Flatten(), # Turns the 2D image into a 1D list of numbers
#     layers.Dense(64, activation='relu'),
#     layers.Dropout(0.5), # Randomly shuts off neurons to prevent "memorization" (Overfitting)
    
#     # Layer 4: Output Layer
#     # Use 'sigmoid' for binary classification (0 or 1)
#     layers.Dense(1, activation='sigmoid')
# ])


from tensorflow.keras import layers, models

model = models.Sequential([
    # Layer 0: Preprocessing
    # This scales pixel values from [0, 255] to [0, 1]
    layers.Rescaling(1./255, input_shape=(128, 44, 1)),
    
    # Layer 1: Convolution + Normalization
    layers.Conv2D(32, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),
    
    # Layer 2: Deeper features
    layers.Conv2D(64, (3, 3), padding='same'),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D((2, 2)),
    
    # Layer 3: Flattening and Dense Layers
    layers.Flatten(), 
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5), 
    
    # Layer 4: Output Layer
    layers.Dense(1, activation='sigmoid')
])

C:\Users\b\AppData\Roaming\Python\Python311\site-packages\keras\src\layers\preprocessing\data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [4]:
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary() # Prints the internal structure of your CNN

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 128, 44, 1)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 44, 32)    │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 44, 32)    │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 128, 44, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 22, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 22, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 22, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 64, 22, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 11, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 22528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │     1,441,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,461,121 (5.57 MB)

 Trainable params: 1,460,929 (5.57 MB)

 Non-trainable params: 192 (768.00 B)

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=20,          # How many times to look at the entire dataset
    batch_size=32,      # How many images to look at before updating weights
    validation_data=(X_val, y_val)
)

Epoch 1/20
437/437 ━━━━━━━━━━━━━━━━━━━━ 40s 87ms/step - accuracy: 0.7787 - loss: 0.6532 - val_accuracy: 0.5000 - val_loss: 6.3888
Epoch 2/20
437/437 ━━━━━━━━━━━━━━━━━━━━ 38s 87ms/step - accuracy: 0.8588 - loss: 0.3196 - val_accuracy: 0.7852 - val_loss: 0.3437
Epoch 3/20
437/437 ━━━━━━━━━━━━━━━━━━━━ 36s 83ms/step - accuracy: 0.8545 - loss: 0.3029 - val_accuracy: 0.9756 - val_loss: 0.0794
Epoch 4/20
437/437 ━━━━━━━━━━━━━━━━━━━━ 39s 89ms/step - accuracy: 0.9138 - loss: 0.2187 - val_accuracy: 0.8312 - val_loss: 0.3600
Epoch 5/20
437/437 ━━━━━━━━━━━━━━━━━━━━ 36s 83ms/step - accuracy: 0.9192 - loss: 0.2022 - val_accuracy: 0.8988 - val_loss: 0.2593
Epoch 6/20
437/437 ━━━━━━━━━━━━━━━━━━━━ 35s 81ms/step - accuracy: 0.9524 - loss: 0.1519 - val_accuracy: 0.9621 - val_loss: 0.1179
Epoch 7/20
437/437 ━━━━━━━━━━━━━━━━━━━━ 36s 83ms/step - accuracy: 0.9532 - loss: 0.1446 - val_accuracy: 0.9653 - val_loss: 0.1009
Epoch 8/20
437/437 ━━━━━━━━━━━━━━━━━━━━ 38s 88ms/step - accuracy: 0.9594 - loss: 0.1279 - 

In [ ]:
import matplotlib.pyplot as plt

# 1. Plot Accuracy
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# 2. Plot Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()

# Testing with the Test Data

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=2)

print(f"\nFinal Test Accuracy: {test_acc*100:.2f}%")

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# 1. Get the model's predictions (probabilities)
y_pred_probs = model.predict(X_test)

# 2. Convert probabilities to hard labels (0 or 1)
# Since we used a sigmoid output, anything > 0.5 is "Fake" (1)
y_pred = (y_pred_probs > 0.5).astype(int)

# 3. Create the Matrix
cm = confusion_matrix(y_test, y_pred)

# 4. Plot it beautifully
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix: Deepfake Detection')
plt.show()

# 5. Print the full report (Precision, Recall, F1-Score)
print(classification_report(y_test, y_pred, target_names=['Real', 'Fake']))